# Optimisation with Differential Evolution (DE)

This notebook is a copy configured to use GEMSEO's Differential Evolution to explore the design space broadly, leveraging the cascade parameterization for blending mandates.

In [ ]:
%matplotlib widget
from aeromaps import create_process
from aeromaps.core.gemseo import CustomDataConverter
import gemseo as gm
from aeromaps.utils.functions import custom_logger_config

custom_logger_config(gm.configure_logger())

## Process, data and compute

In [ ]:
process_is2medium = create_process(
    configuration_file="data/data_optim/config_is2medium.yaml",
    optimisation=True,
)

In [ ]:
# international_ask_share = 66.35 / 100


process_is2medium.parameters.generic_biomass_availability_constraint_enforcement_years = [
    2035,
    2040,
    2045,
    2050,
    2060,
    2070,
]


process_is2medium.parameters.generic_electricity_constraint_enforcement_years = [
    2035,
    2040,
    2045,
    2050,
    2060,
    2070,
]


process_is2medium.parameters.saf_ftg_use_growth_constraint_enforcement_years = [
    2035,
    2040,
    2045,
    2050,
    2060,
    2070,
]
process_is2medium.parameters.saf_co2_use_growth_constraint_enforcement_years = [
    2035,
    2040,
    2045,
    2050,
    2060,
    2070,
]
process_is2medium.parameters.lcaf_use_growth_constraint_enforcement_years = [
    2035,
    2040,
    2045,
    2050,
    2060,
    2070,
]


# Fixed values for the first mandate entries [2020, 2025, 2030]
# These are absolute shares, not fractions
process_is2medium.parameters.saf_ftg_mandate_share_values_fixed = [0, 0, 6.0]
process_is2medium.parameters.saf_co2_mandate_share_values_fixed = [0, 0, 2.0]
process_is2medium.parameters.lcaf_mandate_share_values_fixed = [0, 0, 4.0]


process_is2medium.parameters.rate_ramp_up_constraint_saf_ftg = 0.12


process_is2medium.parameters.rate_ramp_up_constraint_saf_co2 = 0.12


process_is2medium.parameters.rate_ramp_up_constraint_lcaf = 0.12


## Carbon budgets and Carbon Dioxide Removal [GtCO2]
process_is2medium.parameters.net_carbon_budget = 850.0
process_is2medium.parameters.carbon_dioxyde_removal_2100 = 285.0

### Optimisation problem setup with GEMSEO (Differential Evolution)

In [ ]:
from gemseo.algos.design_space import DesignSpace
from gemseo.settings.opt import DIFFERENTIAL_EVOLUTION_Settings

# Create a GEMSEO scenario
process_is2medium.gemseo_settings["scenario_type"] = "MDO"

# Design space: absolute SAF FTG shares and fractions for CO2/LCAF (cascade)
design_space = DesignSpace()
design_space.add_variable(
    "saf_ftg_mandate_share_values_optim",
    size=6,
    lower_bound=[1e-17] * 6,
    upper_bound=[100] * 6,
    value=[8.0, 10.0, 15.0, 15.0, 12.0, 8.0],
)


design_space.add_variable(
    "saf_co2_fraction_optim",
    size=6,
    lower_bound=[1e-17] * 6,
    upper_bound=[1] * 6,
    value=[0.02] * 6,
)


design_space.add_variable(
    "lcaf_fraction_optim",
    size=6,
    lower_bound=[1e-17] * 6,
    upper_bound=[1] * 6,
    value=[0.05] * 6,
)

process_is2medium.gemseo_settings["design_space"] = design_space

# Objective
objective_name = "cumulative_co2_end_year"
process_is2medium.gemseo_settings["objective_name"] = objective_name

# Create scenario
process_is2medium.create_gemseo_scenario()

# Configure Differential Evolution settings via settings model

de_settings = DIFFERENTIAL_EVOLUTION_Settings(
    max_iter=4000,
    popsize=100,
    mutation=(0.5, 1.0),
    recombination=0.7,
    workers=10,
    seed=42,
    polish=True,
    normalize_design_space=True,
)
process_is2medium.gemseo_settings["algorithm"] = de_settings


# Constraints (blend completeness removed - enforced by cascade)
for constraint in [
    "biomass_trajectory_constraint",
    "generic_electricity_trajectory_constraint",
    "saf_co2_use_growth_constraint",
    "saf_ftg_use_growth_constraint",
    "lcaf_use_growth_constraint",
]:
    process_is2medium.scenario.add_constraint(constraint, constraint_type="ineq")

# Differentiation method (not used by DE but kept consistent)
process_is2medium.scenario.set_differentiation_method("finite_differences")

# Ensure GEMSEO converts list-typed variables correctly
CustomDataConverter._list_names.update(process_is2medium.scenario.get_optim_variable_names())

In [ ]:
import warnings

warnings.filterwarnings("ignore")
process_is2medium.compute()

## Results

In [ ]:
process = process_is2medium

In [ ]:
process_is2medium.scenario.post_process(
    post_name="OptHistoryView",
    save=False,
    show=True,
)

In [ ]:
process.plot("emission_factor_per_fuel")

In [ ]:
process.plot("air_transport_co2_emissions")

In [ ]:
from aeromaps.utils.functions import clean_notebooks_on_tests

clean_notebooks_on_tests(globals())